# 02 — Features: sentence graph, ROUGE-L, latency model

Walk through the building blocks the Pareto evaluator depends on.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROC = Path('../data/processed')
sns.set_theme(style='whitegrid')

In [ ]:
docs = pd.read_parquet(PROC / 'docs.parquet')
sample = docs.sample(500, random_state=0).reset_index(drop=True)
sample.head(2)

## Sentence segmentation

In [ ]:
from slm_quant.features import split_sentences
sample['n_sents'] = sample['doc_text'].apply(lambda t: len(split_sentences(t)))
fig, ax = plt.subplots(figsize=(7,3))
sns.histplot(sample['n_sents'], bins=20, ax=ax, color='steelblue')
ax.set_title('Sentences per document')
plt.show()

## Sentence similarity graph (one example)

In [ ]:
from slm_quant.features import sentence_similarity
import networkx as nx
doc = sample.iloc[0]['doc_text']
sents = split_sentences(doc)
sim = sentence_similarity(sents)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(sim, annot=True, fmt='.2f', cmap='Blues', ax=ax)
ax.set_title('Sentence similarity matrix')
plt.show()

## ROUGE-L self-test

In [ ]:
from slm_quant.features import rouge_l
print('identical:', rouge_l('the quick brown fox', 'the quick brown fox'))
print('partial  :', rouge_l('the quick fox', 'the quick brown fox'))
print('disjoint :', rouge_l('apples and oranges', 'cars and bikes'))

## Latency model

In [ ]:
from slm_quant.features import BUDGETS, estimate_latency
rng = np.random.default_rng(0)
lengths = np.arange(50, 520, 20)
rows = []
for tier in BUDGETS:
    for L in lengths:
        rows.append(dict(tier=tier, doc_tokens=L, latency_ms=estimate_latency(int(L), tier, rng=rng)))
df = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8,4))
for tier, sub in df.groupby('tier'):
    ax.plot(sub['doc_tokens'], sub['latency_ms'], label=tier, marker='.')
ax.set_xlabel('doc tokens'); ax.set_ylabel('latency (ms)'); ax.legend(); ax.set_title('Latency vs document length per tier')
plt.show()

## Budget table

In [ ]:
from slm_quant.features import BUDGETS
table = pd.DataFrame([
    {'tier': k, 'max_sents': b.max_sentences, 'max_tokens': b.max_tokens,
     'quant': b.quant, 'size_mb': b.size_mb,
     'alpha_ms': b.alpha_ms, 'beta_ms_per_token': b.beta_ms_per_token}
    for k, b in BUDGETS.items()
])
table

## Takeaways
- Sentence graph is small and dense — PageRank converges in a handful of iterations.
- Latency scales linearly with input length per the analytical model.
- Each budget tier maps to a quantization scheme + an explicit (max_sents, max_tokens) cap.